<a href="https://colab.research.google.com/github/dhaneswaranduraivelu1234-maker/bot/blob/main/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ALBERT Complaint Classifier — Full Dataset (93k rows)
**GPU:** Set Runtime → Change Runtime Type → T4 GPU (free) or A100 (Colab Pro)

**Expected time:** ~45-55 min on T4 (4 epochs) | ~20-25 min on A100

In [ ]:
# ── Step 1: Install packages ──
!pip install -q transformers==4.41.2 datasets==2.19.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 52.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
# ── Step 2: Upload your CSV (run this cell, then click Upload) ──
from google.colab import files
uploaded = files.upload()  # Upload: complaints_text_mapping__3_.csv

Saving complaints_text_mapping__3_.csv.csv to complaints_text_mapping__3_.csv.csv


In [ ]:
import os
# See exactly what Colab named your file
print(os.listdir('/content'))

['.config', 'complaints_text_mapping__3_.csv.csv', 'complaints text mapping (3).csv', 'sample_data']


In [ ]:
# ── Step 3: Verify GPU ──
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# ── Step 4: Imports & Config ──
import os, time, json, random
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)
from transformers import (
    AlbertTokenizer, AlbertForSequenceClassification, get_linear_schedule_with_warmup
)

# ── Config ─────────────────────────────────────────
DATA_PATH = 'complaints_text_mapping__3_.csv.csv'  # adjust if renamed
SAVE_DIR    = './albert_complaint_model'
MODEL_NAME  = 'albert-base-v2'
MAX_LEN     = 128
BATCH_SIZE  = 32   # 32 works on T4/A100; use 16 if OOM
EPOCHS      = 4
LR          = 2e-5
WEIGHT_DECAY= 0.01
WARMUP_RATIO= 0.1
SEED        = 42
# ───────────────────────────────────────────────────

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print('Config ready.')

Config ready.


In [ ]:
# ── Step 5: Load & clean data ──
df = pd.read_csv(DATA_PATH)
df = df[['Issue_Summary', 'Complaint Category Description']].dropna()
df['Issue_Summary'] = df['Issue_Summary'].astype(str).str.strip()
df = df[df['Issue_Summary'] != ''].reset_index(drop=True)

# Remove classes with < 5 samples
counts = df['Complaint Category Description'].value_counts()
valid  = counts[counts >= 5].index
df     = df[df['Complaint Category Description'].isin(valid)].reset_index(drop=True)

print(f'Rows   : {len(df):,}')
print(f'Classes: {df["Complaint Category Description"].nunique()}')
df['Complaint Category Description'].value_counts()

Rows   : 93,606
Classes: 34


,count
Complaint Category Description,
Rates/Charges,20835
Customer Service,9785
Solicit - No Call,8976
Quality of Service,7807
Discontinuance of Service,7073
Non-Juris,6135
Meters,5202
Misc (Non-Juris),4396
Outages,3719


In [ ]:
# ── Step 6: Encode labels & split ──
le           = LabelEncoder()
df['label']  = le.fit_transform(df['Complaint Category Description'])
num_classes  = len(le.classes_)
print(f'Number of classes: {num_classes}')

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

Number of classes: 34
Train: 65,524 | Val: 14,041 | Test: 14,041


In [ ]:
# ── Step 7: Dataset class & tokenizer ──
class ComplaintDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'token_type_ids' : enc['token_type_ids'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long),
        }

print('Tokenizing... (may take ~1 min)')
tokenizer  = AlbertTokenizer.from_pretrained(MODEL_NAME)

train_ds   = ComplaintDataset(train_df['Issue_Summary'], train_df['label'], tokenizer, MAX_LEN)
val_ds     = ComplaintDataset(val_df['Issue_Summary'],   val_df['label'],   tokenizer, MAX_LEN)
test_ds    = ComplaintDataset(test_df['Issue_Summary'],  test_df['label'],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Tokenizing... (may take ~1 min)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

Train batches: 2048 | Val batches: 220


In [ ]:
# ── Step 8: Load model ──
model = AlbertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
model = model.to(device)
total_p = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Params: {total_p/1e6:.1f}M')

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded. Params: 11.7M


In [ ]:
# ── Step 9: Optimizer & scheduler ──
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
print(f'Total steps: {total_steps} | Warmup: {warmup_steps}')

Total steps: 8192 | Warmup: 819


/tmp/ipykernel_8623/1814469621.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler       = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


In [ ]:
# ── Step 10: Training loop ──
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device),
                token_type_ids = batch['token_type_ids'].to(device),
                labels         = batch['label'].to(device)
            )
            total_loss += out.loss.item()
            all_preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    return (
        total_loss / len(loader),
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        all_preds, all_labels
    )

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_f1 = 0.0
history     = []

for epoch in range(EPOCHS):
    model.train()
    total_loss, steps_done = 0.0, 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            out  = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device),
                token_type_ids = batch['token_type_ids'].to(device),
                labels         = batch['label'].to(device)
            )
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        steps_done += 1

        if (step + 1) % 100 == 0:
            elapsed = time.time() - t0
            eta     = (len(train_loader) - step - 1) / (steps_done / elapsed)
            print(f'  Ep {epoch+1}/{EPOCHS} | Step {step+1}/{len(train_loader)} '
                  f'| Loss: {total_loss/steps_done:.4f} | ETA: {eta/60:.1f}min')

    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)
    row = dict(epoch=epoch+1, train_loss=round(total_loss/steps_done,4),
               val_loss=round(val_loss,4), val_acc=round(val_acc,4),
               val_f1=round(val_f1,4), time_min=round((time.time()-t0)/60,1))
    history.append(row)
    print(f'\n── Epoch {epoch+1} ── Train Loss: {row["train_loss"]} | '
          f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | '
          f'Time: {row["time_min"]}min\n')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f'  ✓ Saved best model (val_f1={val_f1:.4f})')

print('Training complete!')

/tmp/ipykernel_8623/3297698905.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


  Ep 1/4 | Step 100/2048 | Loss: 3.2255 | ETA: 9.1min
  Ep 1/4 | Step 200/2048 | Loss: 2.6714 | ETA: 8.4min
  Ep 1/4 | Step 300/2048 | Loss: 2.1218 | ETA: 7.9min
  Ep 1/4 | Step 400/2048 | Loss: 1.6982 | ETA: 7.5min
  Ep 1/4 | Step 500/2048 | Loss: 1.3980 | ETA: 7.1min
  Ep 1/4 | Step 600/2048 | Loss: 1.1839 | ETA: 6.6min
  Ep 1/4 | Step 700/2048 | Loss: 1.0243 | ETA: 6.2min
  Ep 1/4 | Step 800/2048 | Loss: 0.9028 | ETA: 5.7min
  Ep 1/4 | Step 900/2048 | Loss: 0.8064 | ETA: 5.3min
  Ep 1/4 | Step 1000/2048 | Loss: 0.7290 | ETA: 4.9min
  Ep 1/4 | Step 1100/2048 | Loss: 0.6643 | ETA: 4.4min
  Ep 1/4 | Step 1200/2048 | Loss: 0.6103 | ETA: 4.0min
  Ep 1/4 | Step 1300/2048 | Loss: 0.5645 | ETA: 3.5min
  Ep 1/4 | Step 1400/2048 | Loss: 0.5248 | ETA: 3.0min
  Ep 1/4 | Step 1500/2048 | Loss: 0.4903 | ETA: 2.6min
  Ep 1/4 | Step 1600/2048 | Loss: 0.4602 | ETA: 2.1min
  Ep 1/4 | Step 1700/2048 | Loss: 0.4334 | ETA: 1.6min
  Ep 1/4 | Step 1800/2048 | Loss: 0.4094 | ETA: 1.2min
  Ep 1/4 | Step 190

/tmp/ipykernel_8623/3297698905.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


  Ep 2/4 | Step 100/2048 | Loss: 0.0029 | ETA: 9.4min
  Ep 2/4 | Step 200/2048 | Loss: 0.0037 | ETA: 8.9min
  Ep 2/4 | Step 300/2048 | Loss: 0.0033 | ETA: 8.4min
  Ep 2/4 | Step 400/2048 | Loss: 0.0026 | ETA: 8.0min
  Ep 2/4 | Step 500/2048 | Loss: 0.0024 | ETA: 7.5min
  Ep 2/4 | Step 600/2048 | Loss: 0.0021 | ETA: 7.0min
  Ep 2/4 | Step 700/2048 | Loss: 0.0021 | ETA: 6.5min
  Ep 2/4 | Step 800/2048 | Loss: 0.0021 | ETA: 6.0min
  Ep 2/4 | Step 900/2048 | Loss: 0.0020 | ETA: 5.6min
  Ep 2/4 | Step 1000/2048 | Loss: 0.0019 | ETA: 5.1min
  Ep 2/4 | Step 1100/2048 | Loss: 0.0020 | ETA: 4.6min
  Ep 2/4 | Step 1200/2048 | Loss: 0.0019 | ETA: 4.1min
  Ep 2/4 | Step 1300/2048 | Loss: 0.0019 | ETA: 3.6min
  Ep 2/4 | Step 1400/2048 | Loss: 0.0019 | ETA: 3.1min
  Ep 2/4 | Step 1500/2048 | Loss: 0.0019 | ETA: 2.6min
  Ep 2/4 | Step 1600/2048 | Loss: 0.0018 | ETA: 2.2min
  Ep 2/4 | Step 1700/2048 | Loss: 0.0018 | ETA: 1.7min
  Ep 2/4 | Step 1800/2048 | Loss: 0.0017 | ETA: 1.2min
  Ep 2/4 | Step 190

/tmp/ipykernel_8623/3297698905.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


  Ep 3/4 | Step 100/2048 | Loss: 0.0003 | ETA: 9.4min
  Ep 3/4 | Step 200/2048 | Loss: 0.0003 | ETA: 8.9min
  Ep 3/4 | Step 300/2048 | Loss: 0.0003 | ETA: 8.4min
  Ep 3/4 | Step 400/2048 | Loss: 0.0003 | ETA: 7.9min
  Ep 3/4 | Step 500/2048 | Loss: 0.0005 | ETA: 7.5min
  Ep 3/4 | Step 600/2048 | Loss: 0.0006 | ETA: 7.0min
  Ep 3/4 | Step 700/2048 | Loss: 0.0006 | ETA: 6.5min
  Ep 3/4 | Step 800/2048 | Loss: 0.0006 | ETA: 6.0min
  Ep 3/4 | Step 900/2048 | Loss: 0.0008 | ETA: 5.5min
  Ep 3/4 | Step 1000/2048 | Loss: 0.0007 | ETA: 5.1min
  Ep 3/4 | Step 1100/2048 | Loss: 0.0007 | ETA: 4.6min
  Ep 3/4 | Step 1200/2048 | Loss: 0.0007 | ETA: 4.1min
  Ep 3/4 | Step 1300/2048 | Loss: 0.0006 | ETA: 3.6min
  Ep 3/4 | Step 1400/2048 | Loss: 0.0006 | ETA: 3.1min
  Ep 3/4 | Step 1500/2048 | Loss: 0.0006 | ETA: 2.6min
  Ep 3/4 | Step 1600/2048 | Loss: 0.0007 | ETA: 2.2min
  Ep 3/4 | Step 1700/2048 | Loss: 0.0007 | ETA: 1.7min
  Ep 3/4 | Step 1800/2048 | Loss: 0.0007 | ETA: 1.2min
  Ep 3/4 | Step 190

/tmp/ipykernel_8623/3297698905.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


  Ep 4/4 | Step 100/2048 | Loss: 0.0011 | ETA: 9.4min
  Ep 4/4 | Step 200/2048 | Loss: 0.0009 | ETA: 8.9min
  Ep 4/4 | Step 300/2048 | Loss: 0.0006 | ETA: 8.4min
  Ep 4/4 | Step 400/2048 | Loss: 0.0005 | ETA: 7.9min
  Ep 4/4 | Step 500/2048 | Loss: 0.0004 | ETA: 7.5min
  Ep 4/4 | Step 600/2048 | Loss: 0.0004 | ETA: 7.0min
  Ep 4/4 | Step 700/2048 | Loss: 0.0004 | ETA: 6.5min
  Ep 4/4 | Step 800/2048 | Loss: 0.0003 | ETA: 6.0min
  Ep 4/4 | Step 900/2048 | Loss: 0.0003 | ETA: 5.5min
  Ep 4/4 | Step 1000/2048 | Loss: 0.0003 | ETA: 5.1min
  Ep 4/4 | Step 1100/2048 | Loss: 0.0003 | ETA: 4.6min
  Ep 4/4 | Step 1200/2048 | Loss: 0.0003 | ETA: 4.1min
  Ep 4/4 | Step 1300/2048 | Loss: 0.0003 | ETA: 3.6min
  Ep 4/4 | Step 1400/2048 | Loss: 0.0003 | ETA: 3.1min
  Ep 4/4 | Step 1500/2048 | Loss: 0.0003 | ETA: 2.6min
  Ep 4/4 | Step 1600/2048 | Loss: 0.0003 | ETA: 2.2min
  Ep 4/4 | Step 1700/2048 | Loss: 0.0003 | ETA: 1.7min
  Ep 4/4 | Step 1800/2048 | Loss: 0.0003 | ETA: 1.2min
  Ep 4/4 | Step 190

In [ ]:
# ── Step 11: Final test evaluation ──
model = AlbertForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
_, test_acc, test_f1, preds, labels = evaluate(model, test_loader)
p = precision_score(labels, preds, average='weighted', zero_division=0)
r = recall_score(labels,    preds, average='weighted', zero_division=0)

print(f'Accuracy : {test_acc:.4f}')
print(f'Precision: {p:.4f}')
print(f'Recall   : {r:.4f}')
print(f'F1       : {test_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=le.classes_, zero_division=0))

Accuracy : 0.9999
Precision: 0.9999
Recall   : 0.9999
F1       : 0.9999

                                                          precision    recall  f1-score   support

                                                 Billing       1.00      1.00      1.00       100
                                              Commission       1.00      1.00      1.00        12
                                                Cramming       1.00      1.00      1.00        64
                                        Customer Service       1.00      1.00      1.00      1467
                                           Deferred Bill       1.00      1.00      1.00       105
                                        Deposits/Refunds       1.00      1.00      1.00       189
                               Discontinuance of Service       1.00      1.00      1.00      1061
                         Electric Solicitation - No Call       1.00      1.00      1.00         3
                                           I

In [ ]:
# ── Step 12: Save label map & history, then download model ──
pd.DataFrame({'label': range(len(le.classes_)), 'category': le.classes_}).to_csv(
    f'{SAVE_DIR}/label_mapping.csv', index=False)

with open(f'{SAVE_DIR}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Saved. Zipping model folder for download...')
!zip -r albert_complaint_model.zip {SAVE_DIR}
files.download('albert_complaint_model.zip')

Saved. Zipping model folder for download...
  adding: albert_complaint_model/ (stored 0%)
  adding: albert_complaint_model/spiece.model (deflated 49%)
  adding: albert_complaint_model/label_mapping.csv (deflated 40%)
  adding: albert_complaint_model/config.json (deflated 67%)
  adding: albert_complaint_model/training_history.json (deflated 74%)
  adding: albert_complaint_model/special_tokens_map.json (deflated 49%)
  adding: albert_complaint_model/model.safetensors (deflated 7%)
  adding: albert_complaint_model/tokenizer_config.json (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>